# 04 — CatBoost candidate model

The planned deployment candidate. CatBoost accepts categorical columns directly, handles missing numeric values, learns nonlinear interactions, and uses class weights for fraud imbalance.

In [ ]:
!pip -q install catboost pandas pyarrow scikit-learn
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

DATA = Path('/content/drive/MyDrive/ieee_fraud/processed')
ARTIFACTS = Path('/content/drive/MyDrive/ieee_fraud/artifacts'); ARTIFACTS.mkdir(exist_ok=True)
train, valid, test = [pd.read_parquet(DATA / f'{x}.parquet') for x in ['train', 'validation', 'test']]
features = json.loads((DATA / 'preparation_metadata.json').read_text())['feature_columns']
X_train, X_valid = train[features].copy(), valid[features].copy()
y_train, y_valid = train.isFraud, valid.isFraud
cat_columns = X_train.select_dtypes(exclude=np.number).columns.tolist()

# CatBoost needs categorical values represented as non-null strings. Numeric NaN values can remain missing.
for col in cat_columns:
    X_train[col] = X_train[col].fillna('MISSING').astype(str)
    X_valid[col] = X_valid[col].fillna('MISSING').astype(str)
cat_indices = [X_train.columns.get_loc(c) for c in cat_columns]
weight = (y_train == 0).sum() / (y_train == 1).sum()
print('categorical features:', len(cat_columns), 'class weight:', weight)

In [ ]:
model = CatBoostClassifier(
    iterations=2000, learning_rate=0.05, depth=8, loss_function='Logloss', eval_metric='AUC',
    class_weights=[1.0, float(weight)], random_seed=42, verbose=100,
    # Change to 'GPU' only after Colab GPU runtime is enabled and verified.
    task_type='CPU', thread_count=-1, allow_writing_files=False
)
started = time.time()
model.fit(X_train, y_train, cat_features=cat_indices, eval_set=(X_valid, y_valid), early_stopping_rounds=150)
probability = model.predict_proba(X_valid)[:, 1]
metrics = {'model': 'catboost', 'roc_auc': float(roc_auc_score(y_valid, probability)), 'pr_auc': float(average_precision_score(y_valid, probability)), 'training_seconds': round(time.time() - started, 2), 'best_iteration': int(model.get_best_iteration())}
print(metrics)
(ARTIFACTS / 'catboost_metrics.json').write_text(json.dumps(metrics, indent=2))
model.save_model(str(ARTIFACTS / 'fraud_catboost_v1.cbm'))
(ARTIFACTS / 'catboost_feature_schema.json').write_text(json.dumps({'features': features, 'categorical_features': cat_columns}, indent=2))